In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from pathlib import Path

drive_dir=Path("/content/drive/MyDrive/STQD6324_Labour_Shock")
out_dir=drive_dir/"outputs"
core_path=out_dir/"worldbank_core_analysis_cleaned_2015_2024.csv"
df=pd.read_csv(core_path)
print("data shape:",df.shape)
display(df.head())

Mounted at /content/drive
data shape: (720, 18)


,country_code,country_name,is_benchmark,indicator_code,indicator_name,indicator_group,year,value,source,is_missing,value_original,was_missing_originally,value_cleaned,was_imputed,impute_method,missing_position,latest_year_in_series,imputation_assumption
0,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2015,92.540117,World Bank API,False,92.540117,False,92.540117,False,none,not_missing,2024,none
1,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2016,97.213877,World Bank API,False,97.213877,False,97.213877,False,none,not_missing,2024,none
2,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2017,104.073182,World Bank API,False,104.073182,False,104.073182,False,none,not_missing,2024,none
3,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2018,116.229125,World Bank API,False,116.229125,False,116.229125,False,none,not_missing,2024,none
4,CHN,China,True,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),DRI,2019,122.670392,World Bank API,False,122.670392,False,122.670392,False,none,not_missing,2024,none


In [ ]:
#2.check data
print("years:")
print(sorted(df["year"].unique()))

print("countries:")
print(df["country_code"].unique())

print("indicators:")
print(df["indicator_code"].unique())
print("missing value_cleaned:",df["value_cleaned"].isna().sum())
print("row number:",df.shape[0])

years:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
countries:
['CHN' 'IDN' 'KHM' 'LAO' 'MYS' 'PHL' 'SGP' 'THA' 'VNM']
indicators:
['IT.CEL.SETS.P2' 'IT.NET.BBND.P2' 'IT.NET.USER.ZS' 'SL.AGR.EMPL.ZS'
 'SL.IND.EMPL.ZS' 'SL.SRV.EMPL.ZS' 'SL.UEM.1524.ZS' 'SL.UEM.TOTL.ZS']
missing value_cleaned: 0
row number: 720


In [ ]:
#3.create wide data
meta_cols=["country_code","country_name","is_benchmark","year"]

wide_df=df.pivot_table(index=meta_cols,columns="indicator_code",
      values="value_cleaned",aggfunc="first").reset_index()
wide_df.columns.name=None
wide_df=wide_df.rename(columns={"SL.IND.EMPL.ZS":"ind_emp",
                             "SL.AGR.EMPL.ZS":"agr_emp",
                             "SL.SRV.EMPL.ZS":"srv_emp",
                             "SL.UEM.TOTL.ZS":"unemp",
                             "SL.UEM.1524.ZS":"youth_unemp",
                             "IT.NET.USER.ZS":"net_user",
                             "IT.CEL.SETS.P2":"mobile_sub",
                             "IT.NET.BBND.P2":"broadband"})
print("wide data shape:",wide_df.shape)
display(wide_df.head(20))

wide data shape: (90, 12)


,country_code,country_name,is_benchmark,year,mobile_sub,broadband,net_user,agr_emp,ind_emp,srv_emp,youth_unemp,unemp
0,CHN,China,True,2015,92.540117,19.843795,50.299999,28.300472,29.299818,42.399711,10.680,4.650
1,CHN,China,True,2016,97.213877,22.976133,53.200000,27.422125,29.644449,42.933426,10.574,4.560
2,CHN,China,True,2017,104.073182,27.910127,54.300000,26.683583,29.970606,43.345804,10.431,4.470
3,CHN,China,True,2018,116.229125,28.708910,65.358398,25.751495,30.363785,43.884720,9.674,4.310
4,CHN,China,True,2019,122.670392,31.561122,64.080881,24.721990,30.758739,44.519274,10.681,4.560
5,CHN,China,True,2020,120.496715,33.906979,70.052760,23.599860,31.132287,45.267854,12.650,5.000
6,CHN,China,True,2021,121.491918,37.561175,73.053235,22.868778,31.534245,45.596985,12.404,4.550
7,CHN,China,True,2022,124.195249,41.373620,75.611316,24.080109,30.931294,44.988605,14.673,4.980
8,CHN,China,True,2023,128.247014,44.728886,90.600000,22.800877,31.426722,45.772394,15.562,4.670
9,CHN,China,True,2024,131.753932,47.193395,92.000000,22.216498,31.632444,46.151060,15.044,4.590


In [ ]:
#4.check wide data
exp_wide=9*10
act_wide=wide_df.shape[0]
print("expected wide rows:",exp_wide)
print("actual wide rows:",act_wide)

if exp_wide==act_wide:
    print("wide data row number is correct")
else:
    print("check needed: wide data row number mismatch")
print("missing values in wide data:")
display(wide_df.isna().sum())

expected wide rows: 90
actual wide rows: 90
wide data row number is correct
missing values in wide data:


,0
country_code,0
country_name,0
is_benchmark,0
year,0
mobile_sub,0
broadband,0
net_user,0
agr_emp,0
ind_emp,0
srv_emp,0


In [ ]:
#5.min-max normalization
norm_cols=["ind_emp","agr_emp","srv_emp","unemp","youth_unemp",
           "net_user","mobile_sub","broadband"]
dir_dict={"ind_emp":"positive_exposure","agr_emp":"positive_structural_pressure",
    "srv_emp":"robustness_positive_assumption","unemp":"positive_labour_market_pressure",
    "youth_unemp":"positive_youth_labour_pressure","net_user":"positive_readiness",
    "mobile_sub":"positive_readiness","broadband":"positive_readiness"}

note_dict={"ind_emp":"Higher industrial employment may indicate higher automation exposure.",
    "agr_emp":"Higher agricultural employment may indicate structural transformation pressure.",
    "srv_emp":"Service employment share cannot distinguish high-skill vs low-skill service jobs in World Bank data; positive direction is an assumption used only in EVI-robustness, not EVI-main.",
    "unemp":"Higher unemployment indicates weaker labour market absorption.",
    "youth_unemp":"Higher youth unemployment indicates stronger youth labour market pressure.",
    "net_user":"Higher internet use indicates stronger digital readiness.",
    "mobile_sub":"Higher mobile subscriptions indicate stronger digital connectivity.",
    "broadband":"Higher fixed broadband subscriptions indicate stronger digital infrastructure."}

use_dict={"ind_emp":"EVI-main and EVI-robustness","agr_emp":"EVI-main and EVI-robustness",
    "srv_emp":"EVI-robustness only","unemp":"EVI-main and EVI-robustness",
    "youth_unemp":"EVI-main and EVI-robustness","net_user":"DRI","mobile_sub":"DRI",
    "broadband":"DRI"}

norm_rows=[]

for c in norm_cols:
    min_v=wide_df[c].min()
    max_v=wide_df[c].max()

    wide_df[c+"_n"]=(wide_df[c]-min_v)/(max_v-min_v)

    norm_rows.append({"variable":c,"min_value":min_v,"max_value":max_v,
        "normalization":"min-max","direction":dir_dict[c],"index_use":use_dict[c],
        "assumption_note":note_dict[c]})

norm_df=pd.DataFrame(norm_rows)
display(norm_df)
display(wide_df.head())

,variable,min_value,max_value,normalization,direction,index_use,assumption_note
0,ind_emp,6.275821,34.319520,min-max,positive_exposure,EVI-main and EVI-robustness,Higher industrial employment may indicate high...
1,agr_emp,0.079580,72.964953,min-max,positive_structural_pressure,EVI-main and EVI-robustness,Higher agricultural employment may indicate st...
2,srv_emp,20.759226,86.077121,min-max,robustness_positive_assumption,EVI-robustness only,Service employment share cannot distinguish hi...
3,unemp,0.119000,5.000000,min-max,positive_labour_market_pressure,EVI-main and EVI-robustness,Higher unemployment indicates weaker labour ma...
4,youth_unemp,0.378000,17.224000,min-max,positive_youth_labour_pressure,EVI-main and EVI-robustness,Higher youth unemployment indicates stronger y...
5,net_user,6.433191,98.020606,min-max,positive_readiness,DRI,Higher internet use indicates stronger digital...
6,mobile_sub,51.379249,181.221874,min-max,positive_readiness,DRI,Higher mobile subscriptions indicate stronger ...
7,broadband,0.213919,47.193395,min-max,positive_readiness,DRI,Higher fixed broadband subscriptions indicate ...


,country_code,country_name,is_benchmark,year,mobile_sub,broadband,net_user,agr_emp,ind_emp,srv_emp,youth_unemp,unemp,ind_emp_n,agr_emp_n,srv_emp_n,unemp_n,youth_unemp_n,net_user_n,mobile_sub_n,broadband_n
0,CHN,China,True,2015,92.540117,19.843795,50.299999,28.300472,29.299818,42.399711,10.680,4.65,0.821004,0.387196,0.331310,0.928293,0.611540,0.478961,0.317006,0.417839
1,CHN,China,True,2016,97.213877,22.976133,53.200000,27.422125,29.644449,42.933426,10.574,4.56,0.833293,0.375144,0.339481,0.909855,0.605248,0.510625,0.353001,0.484514
2,CHN,China,True,2017,104.073182,27.910127,54.300000,26.683583,29.970606,43.345804,10.431,4.47,0.844924,0.365012,0.345795,0.891416,0.596759,0.522635,0.405829,0.589538
3,CHN,China,True,2018,116.229125,28.708910,65.358398,25.751495,30.363785,43.884720,9.674,4.31,0.858944,0.352223,0.354045,0.858636,0.551822,0.643377,0.499450,0.606541
4,CHN,China,True,2019,122.670392,31.561122,64.080881,24.721990,30.758739,44.519274,10.681,4.56,0.873027,0.338098,0.363760,0.909855,0.611599,0.629428,0.549058,0.667253


In [ ]:
#5.1.save normalization metadata
norm_path=out_dir/"normalization_metadata_2015_2024.csv"
norm_df.to_csv(norm_path,index=False,encoding="utf-8-sig")
print("normalization metadata saved:",norm_path)

normalization metadata saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/normalization_metadata_2015_2024.csv


In [ ]:
#6.check normalized values
norm_check=[]

for c in norm_cols:
    nc=c+"_n"
    norm_check.append({"variable":nc,"min":wide_df[nc].min(),
        "max":wide_df[nc].max(),"missing":wide_df[nc].isna().sum()})
norm_check_df=pd.DataFrame(norm_check)
display(norm_check_df)

,variable,min,max,missing
0,ind_emp_n,0.0,1.0,0
1,agr_emp_n,0.0,1.0,0
2,srv_emp_n,0.0,1.0,0
3,unemp_n,0.0,1.0,0
4,youth_unemp_n,0.0,1.0,0
5,net_user_n,0.0,1.0,0
6,mobile_sub_n,0.0,1.0,0
7,broadband_n,0.0,1.0,0


In [ ]:
#6.1.check mobile_sub range
mobile_rank=wide_df[["country_code","country_name","year","mobile_sub","mobile_sub_n"]].copy()
mobile_rank=mobile_rank.sort_values("mobile_sub",ascending=False)
display(mobile_rank.head(15))

,country_code,country_name,year,mobile_sub,mobile_sub_n
74,THA,Thailand,2019,181.221874,1.000000
77,THA,Thailand,2022,176.222793,0.961499
73,THA,Thailand,2018,175.266002,0.954130
68,SGP,Singapore,2023,173.199933,0.938218
67,SGP,Singapore,2022,173.005291,0.936719
72,THA,Thailand,2017,170.783700,0.919609
69,SGP,Singapore,2024,170.782563,0.919600
71,THA,Thailand,2016,168.881271,0.904957
78,THA,Thailand,2023,168.642808,0.903121
76,THA,Thailand,2021,168.485285,0.901907


In [ ]:
#7.calculate index
wide_df["evi_main"]=(0.30*wide_df["ind_emp_n"]+0.20*wide_df["agr_emp_n"]+
                  0.25*wide_df["unemp_n"]+0.25*wide_df["youth_unemp_n"])

wide_df["evi_robust_010"]=(0.90*wide_df["evi_main"]+0.10*wide_df["srv_emp_n"])

wide_df["evi_robust_020"]=(0.80*wide_df["evi_main"]+0.20*wide_df["srv_emp_n"])

wide_df["dri"]=(wide_df["net_user_n"]+wide_df["mobile_sub_n"]+
                wide_df["broadband_n"])/3

wide_df["risk_gap"]=wide_df["evi_main"]-wide_df["dri"]
wide_df["risk_gap_robust_010"]=wide_df["evi_robust_010"]-wide_df["dri"]
wide_df["risk_gap_robust_020"]=wide_df["evi_robust_020"]-wide_df["dri"]

display(wide_df[["country_code","country_name","is_benchmark","year",
             "evi_main","evi_robust_010","evi_robust_020",
          "dri","risk_gap","risk_gap_robust_010","risk_gap_robust_020"]].head(20))

,country_code,country_name,is_benchmark,year,evi_main,evi_robust_010,evi_robust_020,dri,risk_gap,risk_gap_robust_010,risk_gap_robust_020
0,CHN,China,True,2015,0.708699,0.670960,0.633221,0.404602,0.304097,0.266358,0.228619
1,CHN,China,True,2016,0.703792,0.667361,0.630930,0.449380,0.254412,0.217981,0.181550
2,CHN,China,True,2017,0.698523,0.663250,0.627977,0.506001,0.192522,0.157249,0.121976
3,CHN,China,True,2018,0.680742,0.648073,0.615403,0.583123,0.097620,0.064950,0.032280
4,CHN,China,True,2019,0.709891,0.675278,0.640665,0.615246,0.094645,0.060032,0.025419
5,CHN,China,True,2020,0.762565,0.723831,0.685096,0.648045,0.114520,0.075785,0.037051
6,CHN,China,True,2021,0.738160,0.702370,0.666580,0.687448,0.050712,0.014922,-0.020868
7,CHN,China,True,2022,0.790730,0.748752,0.706773,0.730749,0.059982,0.018003,-0.023975
8,CHN,China,True,2023,0.789835,0.749146,0.708457,0.819509,-0.029673,-0.070362,-0.111051
9,CHN,China,True,2024,0.778648,0.739657,0.700667,0.851093,-0.072446,-0.111436,-0.150427


In [ ]:
#7.1.index formula metadata
formula_rows=[{"index_name":"evi_main",
        "formula":"0.30*ind_emp_n+0.20*agr_emp_n+0.25*unemp_n+0.25*youth_unemp_n",
        "purpose":"Main employment vulnerability index",
        "note":"Service employment is excluded from the main EVI because it does not distinguish high-skill and low-skill service jobs."},
    {"index_name":"evi_robust_010",
        "formula":"0.90*evi_main+0.10*srv_emp_n",
        "purpose":"Robustness check",
        "note":"Tests whether adding service employment at 10% weight changes the ranking while preserving the main EVI structure."},
    {"index_name":"evi_robust_020",
        "formula":"0.80*evi_main+0.20*srv_emp_n",
        "purpose":"Robustness check",
        "note":"Tests whether adding service employment at 20% weight changes the ranking while preserving the main EVI structure."},
    {"index_name":"dri",
        "formula":"(net_user_n+mobile_sub_n+broadband_n)/3",
        "purpose":"Digital readiness index",
        "note":"All three digital readiness indicators are equally weighted."},
    {"index_name":"risk_gap",
        "formula":"evi_main-dri",
        "purpose":"Gap between employment vulnerability and digital readiness",
        "note":"Higher values indicate higher vulnerability relative to readiness."}]
formula_df=pd.DataFrame(formula_rows)
display(formula_df)

formula_path=out_dir/"index_formula_metadata_2015_2024.csv"
formula_df.to_csv(formula_path,index=False,encoding="utf-8-sig")
print("index formula metadata saved:",formula_path)

,index_name,formula,purpose,note
0,evi_main,0.30*ind_emp_n+0.20*agr_emp_n+0.25*unemp_n+0.2...,Main employment vulnerability index,Service employment is excluded from the main E...
1,evi_robust_010,0.90*evi_main+0.10*srv_emp_n,Robustness check,Tests whether adding service employment at 10%...
2,evi_robust_020,0.80*evi_main+0.20*srv_emp_n,Robustness check,Tests whether adding service employment at 20%...
3,dri,(net_user_n+mobile_sub_n+broadband_n)/3,Digital readiness index,All three digital readiness indicators are equ...
4,risk_gap,evi_main-dri,Gap between employment vulnerability and digit...,Higher values indicate higher vulnerability re...


index formula metadata saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/index_formula_metadata_2015_2024.csv


In [ ]:
#8.check index values
index_cols=["evi_main","evi_robust_010","evi_robust_020","dri",
    "risk_gap","risk_gap_robust_010","risk_gap_robust_020"]

index_check=[]

for c in index_cols:
    index_check.append({"index":c,"min":wide_df[c].min(),
        "max":wide_df[c].max(),"missing":wide_df[c].isna().sum()})
index_check_df=pd.DataFrame(index_check)
display(index_check_df)

,index,min,max,missing
0,evi_main,0.281652,0.790730,0
1,evi_robust_010,0.257721,0.749146,0
2,evi_robust_020,0.233515,0.708457,0
3,dri,0.051602,0.851093,0
4,risk_gap,-0.515611,0.467712,0
5,risk_gap_robust_010,-0.446810,0.431544,0
6,risk_gap_robust_020,-0.378010,0.395377,0


In [ ]:
#8.1.save index data
index_path=out_dir/"country_year_evi_dri_index_2015_2024.csv"
wide_df.to_csv(index_path,index=False,encoding="utf-8-sig")
print("country-year index data saved:",index_path)

country-year index data saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/country_year_evi_dri_index_2015_2024.csv


In [ ]:
#8.1.save country-year index data

index_path=out_dir/"country_year_evi_dri_index_2015_2024.csv"

wide_df.to_csv(index_path,index=False,encoding="utf-8-sig")

print("country-year index data saved:",index_path)
print("file exists:",index_path.exists())

check_index_df=pd.read_csv(index_path)
print("saved file shape:",check_index_df.shape)
display(check_index_df.head())

In [ ]:
#9.2024 ranking
idx_2024=wide_df[wide_df["year"]==2024].copy()
idx_2024=idx_2024[["country_code","country_name","is_benchmark","year",
    "evi_main","evi_robust_010","evi_robust_020",
    "dri","risk_gap","risk_gap_robust_010","risk_gap_robust_020"]]

idx_2024=idx_2024.sort_values("risk_gap",ascending=False)
display(idx_2024)

,country_code,country_name,is_benchmark,year,evi_main,evi_robust_010,evi_robust_020,dri,risk_gap,risk_gap_robust_010,risk_gap_robust_020
19,IDN,Indonesia,False,2024,0.599184,0.583655,0.568127,0.457498,0.141686,0.126157,0.110629
39,LAO,Laos,False,2024,0.281652,0.257971,0.234289,0.267443,0.014209,-0.009472,-0.033153
59,PHL,Philippines,False,2024,0.403789,0.422439,0.441090,0.434621,-0.030832,-0.012181,0.006470
49,MYS,Malaysia,False,2024,0.603779,0.610461,0.617142,0.654579,-0.050799,-0.044118,-0.037437
9,CHN,China,True,2024,0.778648,0.739657,0.700667,0.851093,-0.072446,-0.111436,-0.150427
29,KHM,Cambodia,False,2024,0.342318,0.333565,0.324812,0.419831,-0.077513,-0.086266,-0.095019
89,VNM,Vietnam,False,2024,0.535187,0.510829,0.486471,0.645263,-0.110076,-0.134434,-0.158792
79,THA,Thailand,False,2024,0.347706,0.355803,0.363900,0.691902,-0.344196,-0.336099,-0.328002
69,SGP,Singapore,False,2024,0.306826,0.375626,0.444427,0.822436,-0.515611,-0.446810,-0.378010


In [ ]:
#9.1.save 2024 ranking
rank2024_path=out_dir/"index_ranking_2024.csv"
idx_2024.to_csv(rank2024_path,index=False,encoding="utf-8-sig")
print("2024 ranking saved:",rank2024_path)

2024 ranking saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/index_ranking_2024.csv


In [ ]:
#9.2.robustness rank comparison
rank_check=idx_2024.copy()
rank_check["rank_main"]=rank_check["risk_gap"].rank(ascending=False,method="min")
rank_check["rank_robust_010"]=rank_check["risk_gap_robust_010"].rank(ascending=False,method="min")
rank_check["rank_robust_020"]=rank_check["risk_gap_robust_020"].rank(ascending=False,method="min")
rank_check["rank_change_010"]=rank_check["rank_robust_010"]-rank_check["rank_main"]
rank_check["rank_change_020"]=rank_check["rank_robust_020"]-rank_check["rank_main"]

rank_check=rank_check[["country_code","country_name","is_benchmark",
    "rank_main","rank_robust_010","rank_robust_020",
    "rank_change_010","rank_change_020",
    "risk_gap","risk_gap_robust_010","risk_gap_robust_020"]]
rank_check=rank_check.sort_values("rank_main")
display(rank_check)

,country_code,country_name,is_benchmark,rank_main,rank_robust_010,rank_robust_020,rank_change_010,rank_change_020,risk_gap,risk_gap_robust_010,risk_gap_robust_020
19,IDN,Indonesia,False,1.0,1.0,1.0,0.0,0.0,0.141686,0.126157,0.110629
39,LAO,Laos,False,2.0,2.0,3.0,0.0,1.0,0.014209,-0.009472,-0.033153
59,PHL,Philippines,False,3.0,3.0,2.0,0.0,-1.0,-0.030832,-0.012181,0.006470
49,MYS,Malaysia,False,4.0,4.0,4.0,0.0,0.0,-0.050799,-0.044118,-0.037437
9,CHN,China,True,5.0,6.0,6.0,1.0,1.0,-0.072446,-0.111436,-0.150427
29,KHM,Cambodia,False,6.0,5.0,5.0,-1.0,-1.0,-0.077513,-0.086266,-0.095019
89,VNM,Vietnam,False,7.0,7.0,7.0,0.0,0.0,-0.110076,-0.134434,-0.158792
79,THA,Thailand,False,8.0,8.0,8.0,0.0,0.0,-0.344196,-0.336099,-0.328002
69,SGP,Singapore,False,9.0,9.0,9.0,0.0,0.0,-0.515611,-0.446810,-0.378010


In [ ]:
#9.3.save robustness rank comparison
rank_check_path=out_dir/"robustness_rank_comparison_2024.csv"
rank_check.to_csv(rank_check_path,index=False,encoding="utf-8-sig")
print("robustness rank comparison saved:",rank_check_path)

robustness rank comparison saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/robustness_rank_comparison_2024.csv


In [ ]:
#9.4.quick robustness summary
max_change_010=rank_check["rank_change_010"].abs().max()
max_change_020=rank_check["rank_change_020"].abs().max()

print("max rank change when service weight=0.10:",max_change_010)
print("max rank change when service weight=0.20:",max_change_020)
print("robustness rule:")
print("service weight 0.10 allows max rank change <= 1")
print("service weight 0.20 allows max rank change <= 2")

if max_change_010<=1 and max_change_020<=2:
    print("ranking is generally stable under the predefined tolerance rule")
else:
    print("ranking changes exceed the predefined tolerance rule and need further discussion")

max rank change when service weight=0.10: 1.0
max rank change when service weight=0.20: 1.0
robustness rule:
service weight 0.10 allows max rank change <= 1
service weight 0.20 allows max rank change <= 2
ranking is generally stable under the predefined tolerance rule


In [ ]:
#9.5.save robustness rule
robust_rule_rows=[{"check_item":"evi_robust_010","service_weight":0.10,"tolerance_rank_change":1,
        "rule_note":"A maximum rank change of 1 is treated as generally stable under the predefined tolerance rule."},
    {"check_item":"evi_robust_020","service_weight":0.20,"tolerance_rank_change":2,
    "rule_note":"A maximum rank change of 2 is treated as generally stable under the predefined tolerance rule."}]

robust_rule_df=pd.DataFrame(robust_rule_rows)
display(robust_rule_df)
robust_rule_path=out_dir/"robustness_rule_metadata_2024.csv"
robust_rule_df.to_csv(robust_rule_path,index=False,encoding="utf-8-sig")
print("robustness rule metadata saved:",robust_rule_path)

,check_item,service_weight,tolerance_rank_change,rule_note
0,evi_robust_010,0.1,1,A maximum rank change of 1 is treated as gener...
1,evi_robust_020,0.2,2,A maximum rank change of 2 is treated as gener...


robustness rule metadata saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/robustness_rule_metadata_2024.csv


In [ ]:
#10.2024 readiness matrix
matrix_df=idx_2024.copy()
sea_2024=matrix_df[matrix_df["is_benchmark"]==False].copy()
evi_cut=sea_2024["evi_main"].median()
dri_cut=sea_2024["dri"].median()
print("SEA 2024 EVI median:",evi_cut)
print("SEA 2024 DRI median:",dri_cut)

matrix_df["evi_level"]=np.where(matrix_df["evi_main"]>=evi_cut,"High EVI","Low EVI")
matrix_df["dri_level"]=np.where(matrix_df["dri"]>=dri_cut,"High DRI","Low DRI")
def get_quad(row):
    if row["evi_level"]=="High EVI" and row["dri_level"]=="Low DRI":
        return "High vulnerability + Low readiness"
    elif row["evi_level"]=="High EVI" and row["dri_level"]=="High DRI":
        return "High vulnerability + High readiness"
    elif row["evi_level"]=="Low EVI" and row["dri_level"]=="High DRI":
        return "Low vulnerability + High readiness"
    else:
        return "Low vulnerability + Low readiness"

matrix_df["readiness_quadrant"]=matrix_df.apply(get_quad,axis=1)
matrix_df["sample_role"]=np.where(
    matrix_df["is_benchmark"]==True,
    "China benchmark",
    "SEA sample")

matrix_df=matrix_df[["country_code","country_name","is_benchmark","sample_role","year",
    "evi_main","dri","risk_gap","evi_level","dri_level","readiness_quadrant"]]

matrix_df=matrix_df.sort_values(["sample_role","readiness_quadrant","risk_gap"],
    ascending=[False,True,False])
display(matrix_df)

SEA 2024 EVI median: 0.37574734269923216
SEA 2024 DRI median: 0.5513804907606726


,country_code,country_name,is_benchmark,sample_role,year,evi_main,dri,risk_gap,evi_level,dri_level,readiness_quadrant
49,MYS,Malaysia,False,SEA sample,2024,0.603779,0.654579,-0.050799,High EVI,High DRI,High vulnerability + High readiness
89,VNM,Vietnam,False,SEA sample,2024,0.535187,0.645263,-0.110076,High EVI,High DRI,High vulnerability + High readiness
19,IDN,Indonesia,False,SEA sample,2024,0.599184,0.457498,0.141686,High EVI,Low DRI,High vulnerability + Low readiness
59,PHL,Philippines,False,SEA sample,2024,0.403789,0.434621,-0.030832,High EVI,Low DRI,High vulnerability + Low readiness
79,THA,Thailand,False,SEA sample,2024,0.347706,0.691902,-0.344196,Low EVI,High DRI,Low vulnerability + High readiness
69,SGP,Singapore,False,SEA sample,2024,0.306826,0.822436,-0.515611,Low EVI,High DRI,Low vulnerability + High readiness
39,LAO,Laos,False,SEA sample,2024,0.281652,0.267443,0.014209,Low EVI,Low DRI,Low vulnerability + Low readiness
29,KHM,Cambodia,False,SEA sample,2024,0.342318,0.419831,-0.077513,Low EVI,Low DRI,Low vulnerability + Low readiness
9,CHN,China,True,China benchmark,2024,0.778648,0.851093,-0.072446,High EVI,High DRI,High vulnerability + High readiness


In [ ]:
#10.0.save quadrant threshold metadata
quad_rule_rows=[{"threshold_name":"evi_cut","value":evi_cut,"based_on":"2024 SEA sample median",
        "note":"China is excluded from the threshold calculation and used only as a benchmark."},
    {"threshold_name":"dri_cut","value":dri_cut,"based_on":"2024 SEA sample median",
        "note":"This quadrant placement is descriptive and not a statistical significance test."}]

quad_rule_df=pd.DataFrame(quad_rule_rows)
display(quad_rule_df)
quad_rule_path=out_dir/"quadrant_threshold_metadata_2024.csv"
quad_rule_df.to_csv(quad_rule_path,index=False,encoding="utf-8-sig")
print("quadrant threshold metadata saved:",quad_rule_path)

,threshold_name,value,based_on,note
0,evi_cut,0.375747,2024 SEA sample median,China is excluded from the threshold calculati...
1,dri_cut,0.551380,2024 SEA sample median,This quadrant placement is descriptive and not...


quadrant threshold metadata saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/quadrant_threshold_metadata_2024.csv


In [ ]:
#10.1.save readiness matrix
matrix_path=out_dir/"readiness_matrix_2024.csv"
matrix_df.to_csv(matrix_path,index=False,encoding="utf-8-sig")
print("2024 readiness matrix saved:",matrix_path)

2024 readiness matrix saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/readiness_matrix_2024.csv


In [ ]:
#10.2.matrix count
matrix_count=(matrix_df.groupby(["sample_role","readiness_quadrant"])
    .agg(country_count=("country_code","count")).reset_index())

display(matrix_count)
matrix_count_path=out_dir/"readiness_matrix_count_2024.csv"
matrix_count.to_csv(matrix_count_path,index=False,encoding="utf-8-sig")
print("matrix count saved:",matrix_count_path)

,sample_role,readiness_quadrant,country_count
0,China benchmark,High vulnerability + High readiness,1
1,SEA sample,High vulnerability + High readiness,2
2,SEA sample,High vulnerability + Low readiness,2
3,SEA sample,Low vulnerability + High readiness,2
4,SEA sample,Low vulnerability + Low readiness,2


matrix count saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/readiness_matrix_count_2024.csv


In [ ]:
#11.2019-2024 average index
recent_df=wide_df[(wide_df["year"]>=2019)&(wide_df["year"]<=2024)].copy()

avg_df=(recent_df.groupby(["country_code","country_name","is_benchmark"])
    .agg(evi_main_avg=("evi_main","mean"),
        evi_robust_010_avg=("evi_robust_010","mean"),
        evi_robust_020_avg=("evi_robust_020","mean"),
        dri_avg=("dri","mean"),
        risk_gap_avg=("risk_gap","mean"),
        risk_gap_robust_010_avg=("risk_gap_robust_010","mean"),
        risk_gap_robust_020_avg=("risk_gap_robust_020","mean")).reset_index())
avg_df=avg_df.sort_values("risk_gap_avg",ascending=False)
display(avg_df)

,country_code,country_name,is_benchmark,evi_main_avg,evi_robust_010_avg,evi_robust_020_avg,dri_avg,risk_gap_avg,risk_gap_robust_010_avg,risk_gap_robust_020_avg
1,IDN,Indonesia,False,0.624289,0.605431,0.586573,0.424548,0.199741,0.180883,0.162025
3,LAO,Laos,False,0.323994,0.295549,0.267104,0.239553,0.084441,0.055996,0.027550
0,CHN,China,True,0.761638,0.723172,0.684706,0.725348,0.036290,-0.002176,-0.040642
4,MYS,Malaysia,False,0.613802,0.618043,0.622283,0.617094,-0.003292,0.000949,0.005189
5,PHL,Philippines,False,0.422342,0.437251,0.452159,0.472602,-0.050259,-0.035351,-0.020443
2,KHM,Cambodia,False,0.326620,0.318660,0.310701,0.396642,-0.070022,-0.077981,-0.085941
8,VNM,Vietnam,False,0.534807,0.507020,0.479233,0.608690,-0.073882,-0.101669,-0.129457
7,THA,Thailand,False,0.364550,0.368119,0.371689,0.692318,-0.327768,-0.324199,-0.320629
6,SGP,Singapore,False,0.385274,0.445759,0.506244,0.799231,-0.413956,-0.353471,-0.292987


In [ ]:
#11.1.save average index
avg_path=out_dir/"country_average_index_2019_2024.csv"
avg_df.to_csv(avg_path,index=False,encoding="utf-8-sig")
print("2019-2024 average index saved:",avg_path)

2019-2024 average index saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/country_average_index_2019_2024.csv


In [ ]:
#12.SEA-only clustering input
cluster_df=avg_df[avg_df["is_benchmark"]==False].copy()
cluster_df=cluster_df[["country_code","country_name","evi_main_avg",
    "dri_avg","risk_gap_avg"]]
cluster_df=cluster_df.sort_values("risk_gap_avg",ascending=False)
display(cluster_df)

,country_code,country_name,evi_main_avg,dri_avg,risk_gap_avg
1,IDN,Indonesia,0.624289,0.424548,0.199741
3,LAO,Laos,0.323994,0.239553,0.084441
4,MYS,Malaysia,0.613802,0.617094,-0.003292
5,PHL,Philippines,0.422342,0.472602,-0.050259
2,KHM,Cambodia,0.326620,0.396642,-0.070022
8,VNM,Vietnam,0.534807,0.608690,-0.073882
7,THA,Thailand,0.364550,0.692318,-0.327768
6,SGP,Singapore,0.385274,0.799231,-0.413956


In [ ]:
#12.1.save clustering input
cluster_path=out_dir/"sea_clustering_input_2019_2024.csv"
cluster_df.to_csv(cluster_path,index=False,encoding="utf-8-sig")
print("SEA clustering input saved:",cluster_path)

SEA clustering input saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/sea_clustering_input_2019_2024.csv


In [ ]:
#13.exploratory k-means
from sklearn.cluster import KMeans

km_df=cluster_df.copy()
x=km_df[["evi_main_avg","dri_avg"]].copy()
kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
km_df["cluster"]=kmeans.fit_predict(x)
display(km_df.sort_values("cluster"))

,country_code,country_name,evi_main_avg,dri_avg,risk_gap_avg,cluster
7,THA,Thailand,0.364550,0.692318,-0.327768,0
6,SGP,Singapore,0.385274,0.799231,-0.413956,0
3,LAO,Laos,0.323994,0.239553,0.084441,1
5,PHL,Philippines,0.422342,0.472602,-0.050259,1
2,KHM,Cambodia,0.326620,0.396642,-0.070022,1
1,IDN,Indonesia,0.624289,0.424548,0.199741,2
4,MYS,Malaysia,0.613802,0.617094,-0.003292,2
8,VNM,Vietnam,0.534807,0.608690,-0.073882,2


In [ ]:
#13.1.cluster interpretation
cluster_summary=(km_df.groupby("cluster").agg(
        country_count=("country_code","count"),
        avg_evi=("evi_main_avg","mean"),
        avg_dri=("dri_avg","mean"),
        avg_risk_gap=("risk_gap_avg","mean")).reset_index())
cluster_summary=cluster_summary.sort_values("avg_risk_gap",ascending=False)

display(cluster_summary)

,cluster,country_count,avg_evi,avg_dri,avg_risk_gap
2,2,3,0.590966,0.550111,0.040856
1,1,3,0.357652,0.369599,-0.011947
0,0,2,0.374912,0.745774,-0.370862


In [ ]:
#13.2.add cluster label
sea_evi_avg_cut=cluster_df["evi_main_avg"].median()
sea_dri_avg_cut=cluster_df["dri_avg"].median()
print("SEA 2019-2024 average EVI median:",sea_evi_avg_cut)
print("SEA 2019-2024 average DRI median:",sea_dri_avg_cut)

cluster_summary["cluster_label"]=""
for i,r in cluster_summary.iterrows():
    if r["avg_evi"]>=sea_evi_avg_cut and r["avg_dri"]<sea_dri_avg_cut:
        cluster_summary.loc[i,"cluster_label"]="High vulnerability - Low readiness"
    elif r["avg_evi"]>=sea_evi_avg_cut and r["avg_dri"]>=sea_dri_avg_cut:
        cluster_summary.loc[i,"cluster_label"]="High vulnerability - High readiness"
    elif r["avg_evi"]<sea_evi_avg_cut and r["avg_dri"]>=sea_dri_avg_cut:
        cluster_summary.loc[i,"cluster_label"]="Low vulnerability - High readiness"
    else:
        cluster_summary.loc[i,"cluster_label"]="Low vulnerability - Low readiness"
km_df=km_df.merge(cluster_summary[["cluster","cluster_label"]],on="cluster",how="left")

display(cluster_summary)
display(km_df.sort_values(["cluster","risk_gap_avg"],ascending=[True,False]))

SEA 2019-2024 average EVI median: 0.4038082624797934
SEA 2019-2024 average DRI median: 0.5406455588406893


,cluster,country_count,avg_evi,avg_dri,avg_risk_gap,cluster_label
2,2,3,0.590966,0.550111,0.040856,High vulnerability - High readiness
1,1,3,0.357652,0.369599,-0.011947,Low vulnerability - Low readiness
0,0,2,0.374912,0.745774,-0.370862,Low vulnerability - High readiness


,country_code,country_name,evi_main_avg,dri_avg,risk_gap_avg,cluster,cluster_label
6,THA,Thailand,0.364550,0.692318,-0.327768,0,Low vulnerability - High readiness
7,SGP,Singapore,0.385274,0.799231,-0.413956,0,Low vulnerability - High readiness
1,LAO,Laos,0.323994,0.239553,0.084441,1,Low vulnerability - Low readiness
3,PHL,Philippines,0.422342,0.472602,-0.050259,1,Low vulnerability - Low readiness
4,KHM,Cambodia,0.326620,0.396642,-0.070022,1,Low vulnerability - Low readiness
0,IDN,Indonesia,0.624289,0.424548,0.199741,2,High vulnerability - High readiness
2,MYS,Malaysia,0.613802,0.617094,-0.003292,2,High vulnerability - High readiness
5,VNM,Vietnam,0.534807,0.608690,-0.073882,2,High vulnerability - High readiness


In [ ]:
#13.2.1.save cluster label rule
cluster_rule_rows=[{"threshold_name":"sea_evi_avg_cut","value":sea_evi_avg_cut,
        "based_on":"2019-2024 SEA country average median",
        "note":"Used only for interpreting K-means clusters, not for fitting the K-means model."},
    {"threshold_name":"sea_dri_avg_cut","value":sea_dri_avg_cut,
        "based_on":"2019-2024 SEA country average median",
        "note":"China is excluded because K-means clustering is SEA-only."}]

cluster_rule_df=pd.DataFrame(cluster_rule_rows)
display(cluster_rule_df)
cluster_rule_path=out_dir/"cluster_label_threshold_metadata_2019_2024.csv"
cluster_rule_df.to_csv(cluster_rule_path,index=False,encoding="utf-8-sig")

print("cluster label threshold metadata saved:",cluster_rule_path)

,threshold_name,value,based_on,note
0,sea_evi_avg_cut,0.403808,2019-2024 SEA country average median,"Used only for interpreting K-means clusters, n..."
1,sea_dri_avg_cut,0.540646,2019-2024 SEA country average median,China is excluded because K-means clustering i...


cluster label threshold metadata saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/cluster_label_threshold_metadata_2019_2024.csv


In [ ]:
#13.3.save k-means results
km_path=out_dir/"sea_kmeans_cluster_2019_2024.csv"
km_df.to_csv(km_path,index=False,encoding="utf-8-sig")
cluster_sum_path=out_dir/"sea_kmeans_cluster_summary_2019_2024.csv"
cluster_summary.to_csv(cluster_sum_path,index=False,encoding="utf-8-sig")

print("k-means result saved:",km_path)
print("k-means summary saved:",cluster_sum_path)

k-means result saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/sea_kmeans_cluster_2019_2024.csv
k-means summary saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/sea_kmeans_cluster_summary_2019_2024.csv


In [40]:
#13.4.compare quadrant and cluster results
avg_quad_df=cluster_df.copy()
avg_evi_cut=cluster_df["evi_main_avg"].median()
avg_dri_cut=cluster_df["dri_avg"].median()

avg_quad_df["avg_evi_level"]=np.where(avg_quad_df["evi_main_avg"]>=avg_evi_cut,
    "High EVI","Low EVI")
avg_quad_df["avg_dri_level"]=np.where(avg_quad_df["dri_avg"]>=avg_dri_cut,
    "High DRI","Low DRI")

def get_avg_quad(row):
    if row["avg_evi_level"]=="High EVI" and row["avg_dri_level"]=="Low DRI":
        return "High vulnerability + Low readiness"
    elif row["avg_evi_level"]=="High EVI" and row["avg_dri_level"]=="High DRI":
        return "High vulnerability + High readiness"
    elif row["avg_evi_level"]=="Low EVI" and row["avg_dri_level"]=="High DRI":
        return "Low vulnerability + High readiness"
    else:
        return "Low vulnerability + Low readiness"

avg_quad_df["avg_readiness_quadrant"]=avg_quad_df.apply(get_avg_quad,axis=1)
matrix_sea=matrix_df[matrix_df["is_benchmark"]==False][["country_code",
    "readiness_quadrant"]].copy()

matrix_sea=matrix_sea.rename(columns={"readiness_quadrant":"quadrant_2024"})

compare_df=avg_quad_df.merge(km_df[["country_code","cluster","cluster_label"]],
    on="country_code",how="left")
compare_df=compare_df.merge(matrix_sea,on="country_code",how="left")
compare_df=compare_df[["country_code","country_name",
                       "evi_main_avg","dri_avg","risk_gap_avg","avg_readiness_quadrant",
                       "quadrant_2024","cluster","cluster_label"]]
compare_df=compare_df.sort_values("risk_gap_avg",ascending=False)

display(compare_df)

,country_code,country_name,evi_main_avg,dri_avg,risk_gap_avg,avg_readiness_quadrant,quadrant_2024,cluster,cluster_label
0,IDN,Indonesia,0.624289,0.424548,0.199741,High vulnerability + Low readiness,High vulnerability + Low readiness,2,High vulnerability - High readiness
1,LAO,Laos,0.323994,0.239553,0.084441,Low vulnerability + Low readiness,Low vulnerability + Low readiness,1,Low vulnerability - Low readiness
2,MYS,Malaysia,0.613802,0.617094,-0.003292,High vulnerability + High readiness,High vulnerability + High readiness,2,High vulnerability - High readiness
3,PHL,Philippines,0.422342,0.472602,-0.050259,High vulnerability + Low readiness,High vulnerability + Low readiness,1,Low vulnerability - Low readiness
4,KHM,Cambodia,0.326620,0.396642,-0.070022,Low vulnerability + Low readiness,Low vulnerability + Low readiness,1,Low vulnerability - Low readiness
5,VNM,Vietnam,0.534807,0.608690,-0.073882,High vulnerability + High readiness,High vulnerability + High readiness,2,High vulnerability - High readiness
6,THA,Thailand,0.364550,0.692318,-0.327768,Low vulnerability + High readiness,Low vulnerability + High readiness,0,Low vulnerability - High readiness
7,SGP,Singapore,0.385274,0.799231,-0.413956,Low vulnerability + High readiness,Low vulnerability + High readiness,0,Low vulnerability - High readiness


In [41]:
#13.5.save comparison table
compare_path=out_dir/"quadrant_cluster_comparison_2019_2024.csv"
compare_df.to_csv(compare_path,index=False,encoding="utf-8-sig")
print("quadrant and cluster comparison saved:",compare_path)

quadrant and cluster comparison saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/quadrant_cluster_comparison_2019_2024.csv


In [43]:
#13.6.check agreement between quadrant and cluster label
compare_df["method_agreement"]=np.where(compare_df["avg_readiness_quadrant"].str.replace(" + "," - ",regex=False)==compare_df["cluster_label"],
    "Same","Different")

agreement_summary=(compare_df.groupby("method_agreement")
    .agg(country_count=("country_code","count")).reset_index())
display(compare_df)
display(agreement_summary)
agreement_path=out_dir/"quadrant_cluster_agreement_2019_2024.csv"
compare_df.to_csv(agreement_path,index=False,encoding="utf-8-sig")

agreement_summary_path=out_dir/"quadrant_cluster_agreement_summary_2019_2024.csv"
agreement_summary.to_csv(agreement_summary_path,index=False,encoding="utf-8-sig")
print("agreement table saved:",agreement_path)
print("agreement summary saved:",agreement_summary_path)

,country_code,country_name,evi_main_avg,dri_avg,risk_gap_avg,avg_readiness_quadrant,quadrant_2024,cluster,cluster_label,method_agreement
0,IDN,Indonesia,0.624289,0.424548,0.199741,High vulnerability + Low readiness,High vulnerability + Low readiness,2,High vulnerability - High readiness,Different
1,LAO,Laos,0.323994,0.239553,0.084441,Low vulnerability + Low readiness,Low vulnerability + Low readiness,1,Low vulnerability - Low readiness,Same
2,MYS,Malaysia,0.613802,0.617094,-0.003292,High vulnerability + High readiness,High vulnerability + High readiness,2,High vulnerability - High readiness,Same
3,PHL,Philippines,0.422342,0.472602,-0.050259,High vulnerability + Low readiness,High vulnerability + Low readiness,1,Low vulnerability - Low readiness,Different
4,KHM,Cambodia,0.326620,0.396642,-0.070022,Low vulnerability + Low readiness,Low vulnerability + Low readiness,1,Low vulnerability - Low readiness,Same
5,VNM,Vietnam,0.534807,0.608690,-0.073882,High vulnerability + High readiness,High vulnerability + High readiness,2,High vulnerability - High readiness,Same
6,THA,Thailand,0.364550,0.692318,-0.327768,Low vulnerability + High readiness,Low vulnerability + High readiness,0,Low vulnerability - High readiness,Same
7,SGP,Singapore,0.385274,0.799231,-0.413956,Low vulnerability + High readiness,Low vulnerability + High readiness,0,Low vulnerability - High readiness,Same


,method_agreement,country_count
0,Different,2
1,Same,6


agreement table saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/quadrant_cluster_agreement_2019_2024.csv
agreement summary saved: /content/drive/MyDrive/STQD6324_Labour_Shock/outputs/quadrant_cluster_agreement_summary_2019_2024.csv


In [44]:
#14.final output check
output_files=[
    "normalization_metadata_2015_2024.csv",
    "index_formula_metadata_2015_2024.csv",
    "country_year_evi_dri_index_2015_2024.csv",
    "index_ranking_2024.csv",
    "robustness_rank_comparison_2024.csv",
    "robustness_rule_metadata_2024.csv",
    "readiness_matrix_2024.csv",
    "readiness_matrix_count_2024.csv",
    "country_average_index_2019_2024.csv",
    "sea_clustering_input_2019_2024.csv",
    "sea_kmeans_cluster_2019_2024.csv",
    "sea_kmeans_cluster_summary_2019_2024.csv",
    "quadrant_threshold_metadata_2024.csv",
"cluster_label_threshold_metadata_2019_2024.csv",
    "quadrant_cluster_comparison_2019_2024.csv",
    "quadrant_cluster_agreement_2019_2024.csv",
"quadrant_cluster_agreement_summary_2019_2024.csv"]

for f in output_files:
    p=out_dir/f
    print(f,p.exists())

normalization_metadata_2015_2024.csv True
index_formula_metadata_2015_2024.csv True
country_year_evi_dri_index_2015_2024.csv True
index_ranking_2024.csv True
robustness_rank_comparison_2024.csv True
robustness_rule_metadata_2024.csv True
readiness_matrix_2024.csv True
readiness_matrix_count_2024.csv True
country_average_index_2019_2024.csv True
sea_clustering_input_2019_2024.csv True
sea_kmeans_cluster_2019_2024.csv True
sea_kmeans_cluster_summary_2019_2024.csv True
quadrant_threshold_metadata_2024.csv True
cluster_label_threshold_metadata_2019_2024.csv True
quadrant_cluster_comparison_2019_2024.csv True
quadrant_cluster_agreement_2019_2024.csv True
quadrant_cluster_agreement_summary_2019_2024.csv True
